# Cleaning decisions

**Purpose:**  `initial_eda.ipynb` already established what is in
the files. This notebook exists to settle the handful of judgement calls that `clean.py` needs.

**How to use it:** for each decision, run the checks, then write the rule in the
*Decision* cell in one sentence. Those sentences get copied into `clean.py` as comments and
into the README. Nothing in this notebook produces project output.

**Open decisions:**
1. Which audio row to keep when a `song_id` has more than one
2. What to do with conflicting chart rows (same week + rank, or same week + song)
3. Whether to flag or drop implausible durations
4. How to map 1,145 Spotify genre labels into a few buckets
5. Which date separates the streaming era (verified externally, not from the data)


In [ ]:
from pathlib import Path
import ast
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
billboard = pd.read_csv(RAW / "billboard.csv")
audio = pd.read_csv(RAW / "audio_features.csv")

billboard["chart_date"] = pd.to_datetime(billboard.week_id, format="%m/%d/%Y")


def parse_genres(value):
    """"['rock', 'pop']" -> ['rock', 'pop']; missing or unparseable -> []."""
    if pd.isna(value):
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []


print(f"billboard: {len(billboard):,} rows")
print(f"audio    : {len(audio):,} rows, {audio.song_id.nunique():,} unique song_ids")

---
## Decision 1 — which audio row do we keep?

**Why it matters:** the merge is `billboard.merge(audio, on="song_id", how="left")`. If `audio`
has two rows for a song, every chart week of that song is duplicated and nothing raises an
error. So `audio` must have exactly one row per `song_id` *before* the merge.

**What was already found** (`initial_eda.ipynb`, cell 4): 24 exact duplicate rows, and 117 rows
beyond one per `song_id`.

### 1a. Split the problem in two

Exact duplicates need no decision — drop them. Only rows that *disagree* need a rule.


In [ ]:
exact_dupes = audio.duplicated().sum()
audio_nodupes = audio.drop_duplicates()
conflicting_ids = audio_nodupes.duplicated(subset="song_id").sum()

print("exact duplicate rows:", exact_dupes)
print("song_ids with genuinely conflicting rows:", conflicting_ids)

exact duplicate rows: 24
song_ids with genuinely conflicting rows: 93


0    False
1    False
2    False
3    False
4    False
dtype: bool

### 1b. What do the conflicting rows disagree about?

This is the question that decides the rule. Two very different cases:

- they differ only on **metadata** (`spotify_track_id`, album, popularity, preview URL) —
  same recording listed twice, so either row is fine and you pick by a tie-break
- they differ on the **audio measurements** themselves (`valence`, `energy`, `tempo`, duration) —
  genuinely different recordings (a remix, a live version, a remaster) sharing one `song_id`


In [ ]:
# The columns that end up in the cleaned data. A difference outside this list does not matter.
AUDIO_COLS = [
    "spotify_track_id",
    "spotify_track_duration_ms",
    "spotify_track_explicit",
    "spotify_track_popularity",
    "spotify_genre",
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "loudness",
    "tempo",
    "key",
    "mode",
    "time_signature",
]

# Fallback so this cell runs even if 1a has not been executed yet.
if not isinstance(globals().get("audio_nodupes"), pd.DataFrame):
    audio_nodupes = audio.drop_duplicates()

# One row per song, one column per field: how many DISTINCT values does this song have?
# dropna=False matters -- "0.5 in one row, missing in the other" is a real difference,
# and the default would silently report agreement.
nun = audio_nodupes.groupby("song_id")[AUDIO_COLS].nunique(dropna=False)

conflicts = nun[nun.max(axis=1) > 1]
print("songs whose rows disagree somewhere:", len(conflicts))

print("\ndisagreements per field:")
print((conflicts > 1).sum().sort_values(ascending=False).loc[lambda s: s > 0])

# Popularity is bookkeeping, not a measurement, so separate it from the rest.
SUBSTANTIVE = [c for c in AUDIO_COLS if c != "spotify_track_popularity"]
substantive = conflicts[(conflicts[SUBSTANTIVE] > 1).any(axis=1)]
print("\nof those, disagreeing on something other than popularity:", len(substantive))

In [ ]:
# The rows that disagree on something substantive, pairs on adjacent lines.
LOOK = ["song_id", "spotify_track_id", "spotify_track_album", "spotify_track_duration_ms",
        "spotify_track_popularity", "danceability", "energy", "valence", "tempo", "spotify_genre"]

display(audio_nodupes[audio_nodupes.song_id.isin(substantive.index)]
        .sort_values("song_id")[LOOK])

# The question that decides the rule: same recording twice, or two different recordings?
conflicting_rows = audio_nodupes[audio_nodupes.song_id.isin(conflicts.index)]
one_track_each = conflicting_rows.groupby("song_id").spotify_track_id.nunique(dropna=False).eq(1).all()
print("every conflicting pair shares a single spotify_track_id:", one_track_each)

# If it is the same recording, how far apart can the numbers be?
AUDIO_ONLY = ["danceability", "energy", "valence", "acousticness", "instrumentalness",
              "liveness", "speechiness", "loudness", "tempo"]
spread = conflicting_rows.groupby("song_id")[AUDIO_ONLY].agg(lambda s: s.max() - s.min())
print("largest disagreement in any single audio value:", round(spread.max().max(), 4))

### 1c. Test the tie-break you are considering

A plausible rule is *keep the row with the highest `spotify_track_popularity`*, on the reasoning
that the most-streamed version is most likely the hit. Check it behaves before adopting it:

- does every conflicting song have a non-missing popularity value, or would some ties stay unbroken?
- when popularity ties, what breaks the tie? (a deterministic fallback is needed so two runs of
  `clean.py` never disagree — e.g. then take the longer duration, then the first `spotify_track_id`
  alphabetically)


In [ ]:

print("conflicting rows with a missing popularity value:",
      conflicting_rows.spotify_track_popularity.isna().sum())
print("songs still tied on maximum popularity:",
      (conflicting_rows.groupby("song_id").spotify_track_popularity
       .agg(lambda s: (s == s.max()).sum()) > 1).sum())

audio_resolved = (
    audio_nodupes
    .assign(_n_genres=audio_nodupes.spotify_genre.map(lambda g: len(parse_genres(g))))
    .sort_values(["song_id", "spotify_track_popularity", "_n_genres", "spotify_track_id"],
                 ascending=[True, False, False, True])   # missing values sink to the bottom
    .drop_duplicates("song_id", keep="first")
    .drop(columns="_n_genres")
)

assert audio_resolved.song_id.is_unique, "still duplicated after the rule"
print(f"\nrows: {len(audio):,} raw -> {len(audio_nodupes):,} after exact duplicates"
      f" -> {len(audio_resolved):,} resolved")

# The merge is now safe to attempt. validate="m:1" raises if the right side is not unique,
# and a left join onto unique keys must not change the row count.
merged = billboard.merge(audio_resolved, on="song_id", how="left", validate="m:1")
assert len(merged) == len(billboard), "the merge changed the number of chart rows"
print(f"merge ok: {merged.valence.notna().mean():.1%} of chart rows have audio features")

> ### ✍️ Decision 1
>
> **Rule:** drop rows that are exact duplicates across every column; for the songs that remain
> duplicated, keep the row with the highest `spotify_track_popularity`, breaking ties by the
> longer genre list and then by `spotify_track_id` alphabetically.
>
> **Affects:** 24 exact duplicate rows, plus ~93 songs needing the tie-break — about 0.6% of
> chart rows once joined. _(confirm these against your own output)_
>
> **Why:** every conflicting pair shares the same `spotify_track_id`, so these are not different
> recordings but the same track captured at different times. Most pairs differ only in
> popularity; 9 differ in Spotify's artist genre tags, and 2 differ in audio values by at most
> 0.026 on the 0-1 scores (0.12 dB in loudness). No choice here can move a result, so the rule exists to make the outcome deterministic
> rather than to find the "true" row.
>
> **Worth keeping for the blog:** Spotify returned slightly different audio values for the same
> track on different scrapes — a concrete illustration that these are model estimates, not fixed
> measurements.

---
## Decision 2 — repeated chart rows

Two different patterns hide behind "duplicates" here, and they need different answers:

- **A:** the *same song* listed twice in one chart week
- **B:** two *different songs* sharing one rank in the same week

**Why it matters:** pattern A would double-count a song in any per-week figure and inflate its
chart longevity. Pattern B only matters if some part of the pipeline assumes a rank identifies
a song.

In [ ]:
# Pattern A: the same song twice in one chart week.
same_song = billboard[billboard.duplicated(["week_id", "song_id"], keep=False)]
print(f"A - same song twice in a week: {len(same_song)} rows, {same_song.song_id.nunique()} song(s)")
display(same_song.sort_values(["song_id", "chart_date", "week_position"])
        [["chart_date", "week_position", "song", "performer", "instance",
          "peak_position", "weeks_on_chart"]].head(8))

# Pattern B: two different songs sharing one rank.
same_rank = billboard[billboard.duplicated(["week_id", "week_position"], keep=False)]
print(f"\nB - two songs sharing a rank: {len(same_rank)} rows, "
      f"{same_rank.week_id.nunique()} weeks, {same_rank.song_id.nunique()} songs")
display(same_rank.sort_values(["chart_date", "week_position"])
        [["chart_date", "week_position", "song", "performer", "weeks_on_chart"]].head(8))

In [ ]:
# Is pattern B concentrated in one era? If it were, removing those rows would bias that era.
print("pattern B rows by decade:")
print(same_rank.chart_date.dt.year.floordiv(10).mul(10).value_counts().sort_index().to_string())

# Weeks that do not contain exactly 100 rows.
week_sizes = billboard.groupby("chart_date").size()
print("\nweeks without 100 rows:")
print(week_sizes[week_sizes != 100].to_string())

# Pattern A, up close: the source weeks_on_chart advances TWICE in a single week,
# because it counts rows rather than weeks. This is why chart history is recomputed.
sid = same_song.song_id.iloc[0]
song_rows = billboard[billboard.song_id == sid]
print(f"\n{sid}")
print(f"  chart rows            : {len(song_rows)}")
print(f"  distinct chart dates  : {song_rows.chart_date.nunique()}   <- what we count")
print(f"  best position reached : {song_rows.week_position.min()}")
print(f"  source weeks_on_chart : {song_rows.weeks_on_chart.max()}   <- not usable")

# Be explicit about which keys hold and which do not.
print("\n(song_id, chart_date) pairs that repeat:",
      billboard.duplicated(["song_id", "chart_date"]).sum(),
      "-> chart_entries.csv keeps both rows, songs.csv counts distinct dates")

> ### ✍️ Decision 2
>
> **Rule:** keep every chart row. Both patterns are genuine chart entries, not errors. Song-level
> figures count *distinct chart dates* rather than rows, and the best rank is the minimum position
> reached, so a song occupying two positions in one week is never counted twice. No validation
> assumes one row per rank per week, or exactly 100 rows per week.
>
> **Affects:** pattern A is 26 rows belonging to a single song; pattern B is 58 rows across 24
> weeks, scattered between 1960 and 1989, or 0.018% of chart rows.
>
> **Why:** pattern A is "Unchained Melody" by The Righteous Brothers in late 1990, when two
> separate recordings charted at once after the film *Ghost* revived the song; `song_id` is built
> from title and performer, so the two recordings share one identifier. Pattern B is two different
> songs tying on a rank. Deleting either would remove real chart history to satisfy an assumption
> the data never made.
>
> **Follows from this:** the source `weeks_on_chart` advances twice in a single week during the
> 1990 overlap, because it counts rows rather than weeks, and it also keeps counting across
> separate chart runs. All chart-history figures are therefore recomputed from the weekly records,
> and `peak_position` and `weeks_on_chart` are not carried into the cleaned data.

---
## Decision 3 — implausible durations

The raw file contains a 51-minute "track". The question is not whether the value is odd, but
whether it can change an answer. Test that rather than arguing about a threshold.

In [ ]:
audio_dur = audio.drop_duplicates("song_id").copy()
audio_dur["duration_min"] = audio_dur.spotify_track_duration_ms / 60_000

print("tracks over 10 minutes:", (audio_dur.duration_min > 10).sum(),
      "| under 1 minute:", (audio_dur.duration_min < 1).sum(),
      "| of", audio_dur.duration_min.notna().sum(), "with a duration")

# Are these actually charting songs, or audio rows nothing points at?
chart_history = billboard.groupby("song_id").agg(
    chart_weeks=("chart_date", "nunique"),
    best_rank=("week_position", "min"),
    first_charted=("chart_date", "min"))
audio_dur = audio_dur.join(chart_history, on="song_id")

long_tracks = audio_dur[audio_dur.duration_min > 10]
print("\nlong tracks that really charted:", long_tracks.chart_weeks.notna().sum(), "of", len(long_tracks))
print("chart weeks they account for:", int(long_tracks.chart_weeks.sum()),
      f"({long_tracks.chart_weeks.sum() / len(billboard):.2%} of chart rows)")

# Read the album column: this is the single-versus-album-version problem, not a bad number.
display(long_tracks.sort_values("duration_min", ascending=False)
        [["song", "performer", "duration_min", "chart_weeks", "best_rank",
          "first_charted", "spotify_track_album"]].head(10))

> ### ✍️ Decision 3
>
> **Rule:** no rows are removed or flagged on duration. Every figure reports the median, and
> excluding tracks over 10 minutes moves a decade median by at most 0.003 minutes, under a fifth
> of a second.
>
> **Affects:** 28 tracks over 10 minutes out of 24,288 with a duration, together accounting for
> 305 chart weeks, or 0.09% of chart rows. All 28 are genuine charting songs.
>
> **Why:** the median is the middle song, so a single long outlier cannot pull it. The mean does
> move, which is why medians are used throughout.
>
> **The more important finding, for the blog rather than for cleaning:** these are not wrong
> durations, they are the wrong *version* of the right song. "Tubular Bells" charted as an edited
> single but is matched to the 26-minute album track; "Autobahn" is matched to a 22-minute
> remaster; "Sexual" by Goddess is matched to a meditation album and is simply the wrong
> recording. So `duration_ms` describes whichever recording Spotify matched, not necessarily the
> one that charted — which bears directly on the "songs got shorter" thread, since duration is the
> feature most exposed to version mismatches. Album versions are a bigger factor for older songs,
> so the bias is not evenly spread across decades.

---
## Decision 4 — genre buckets

`spotify_genre` holds a *string* that looks like a list: `"['british invasion', 'rock']"`.
1,145 distinct labels, a median of 4 per song, and 14% of songs carry none.

Two things to keep in mind:

- the labels describe the **artist**, not the song, and reflect Spotify's **present-day** tagging
- a song carries several labels, so a **priority order** is needed, and that order is a choice we
  made rather than a fact in the data

**The target categories are Billboard's own**, taken from the genre charts they publish at
<https://www.billboard.com/charts/>: Hot Country Songs, Hot R&B/Hip-Hop Songs (split there into
Hot R&B Songs and Hot Rap Songs), Hot Rock & Alternative Songs, Hot Latin Songs, Hot
Dance/Electronic Songs, and Hot Christian and Gospel Songs. Pop is taken from their Pop Airplay
chart. Using Billboard's categories on Billboard's chart means the taxonomy is theirs, not ours.

One category is ours and has to be declared as such: **traditional pop / standards**, for
"adult standards" (3,715 songs) and "brill building pop" (3,302), which dominate the 1950s and
60s. Billboard publishes no modern chart for that music, but without a bucket those decades
would empty into "other".

In [ ]:
# parse_genres() is defined in the setup cell at the top of the notebook.
from collections import Counter

label_counts = Counter(
    label for labels in audio.spotify_genre.map(parse_genres) for label in labels)

print(f"{len(label_counts):,} distinct labels\n")
print("most common 40:")
for label, count in label_counts.most_common(40):
    print(f"  {count:>5}  {label}")

In [ ]:
# Billboard's categories. A song carries several labels, so the bucket is decided by a VOTE:
# every label that matches a bucket casts one vote, and the bucket with the most votes wins.
# Ties are broken by the order below, which is why "pop" sits last -- it attaches to almost
# every charting artist and should never win a tie.
#
# A vote rather than "first match wins" because one stray label should not outrank five:
# Chuck Berry carries blues rock, classic rock, rock, rock-and-roll, rockabilly AND soul.
GENRE_RULES = [
    ("country",      ["country", "nashville", "redneck", "bluegrass"]),
    ("rap",          ["rap", "hip hop", "trap", "drill", "grime"]),
    ("r&b",          ["r&b", "rhythm and blues", "soul", "motown", "funk", "quiet storm",
                      "urban contemporary", "new jack swing", "doo-wop"]),
    ("latin",        ["latin", "reggaeton", "salsa", "bachata", "merengue",
                      "regional mexican", "banda", "tejano", "spanish"]),
    ("christian",    ["christian", "gospel", "worship", "ccm"]),
    ("dance",        ["edm", "house", "techno", "trance", "electronic", "disco",
                      "eurodance", "freestyle", "hi-nrg"]),
    ("rock",         ["rock", "metal", "punk", "grunge", "new wave", "mellow gold",
                      "british invasion", "merseybeat", "psychedelic"]),
    ("traditional",  ["adult standards", "brill building", "easy listening", "lounge",
                      "big band", "swing", "vocal jazz", "jazz", "blues", "folk"]),
    ("pop",          ["pop", "boy band", "girl group", "bubblegum"]),
]


def to_bucket(labels):
    """The Billboard category with the most matching labels; ties go to the earlier rule."""
    if not labels:
        return "unlabeled"
    votes = {bucket: sum(any(k in label for k in keywords) for label in labels)
             for bucket, keywords in GENRE_RULES}
    best = max(votes.values())
    if best == 0:
        return "other"
    for bucket, _ in GENRE_RULES:          # priority order breaks ties
        if votes[bucket] == best:
            return bucket


genres = audio.drop_duplicates("song_id")[["song_id", "song", "performer", "spotify_genre"]].copy()
genres["labels"] = genres.spotify_genre.map(parse_genres)
genres["genre_bucket"] = genres.labels.map(to_bucket)

print(genres.genre_bucket.value_counts().to_string())
print("\nshare in a real Billboard category:",
      f"{genres.genre_bucket.isin([b for b, _ in GENRE_RULES]).mean():.1%}")

In [ ]:
# What still falls through? If a frequent label lands in "other", it needs a keyword.
leftover = Counter(label for labels in genres.loc[genres.genre_bucket == "other", "labels"]
                   for label in labels)
print("most common labels still unmatched:")
for label, count in leftover.most_common(15):
    print(f"  {count:>4}  {label}")

# Does the mix change over time in a way that matches music history?
first_year = billboard.groupby("song_id").chart_date.min().dt.year
genres = genres.join(first_year.rename("first_year"), on="song_id")
window = genres[genres.first_year.between(1959, 2020)]
mix = (pd.crosstab(window.first_year // 10 * 10, window.genre_bucket, normalize="index") * 100)
display(mix.round(1))

# The sanity check: songs we can judge by ear.
KNOWN = ["Billie Jean", "Smells Like Teen Spirit", "Jolene", "Respect", "Bad Guy",
         "Rapper's Delight", "Despacito", "Stayin' Alive", "Johnny B. Goode", "Mack The Knife"]
display(genres[genres.song.isin(KNOWN)][["song", "performer", "genre_bucket", "spotify_genre"]])

> ### ✍️ Decision 4
>
> **Categories:** Billboard's own, from the genre charts at <https://www.billboard.com/charts/> —
> country, rap, r&b, latin, christian, dance/electronic, rock, pop — plus one category of our own,
> **traditional**, for the "adult standards" and "brill building pop" music that dominates the
> 1950s and 60s and has no modern Billboard chart.
>
> **Rule:** every Spotify label that matches a category casts one vote, and the category with the
> most votes wins. Ties are broken by the order in `GENRE_RULES`, where pop sits last because it
> attaches to nearly every charting artist. Songs with no labels become `unlabeled`; songs whose
> labels match nothing become `other`.
>
> **Why a vote and not "first match wins":** the first-match version put Chuck Berry's "Johnny B.
> Goode" in r&b, because a single `soul` label outranked `blues rock`, `classic rock`, `rock`,
> `rock-and-roll` and `rockabilly`. It also sent Billie Eilish's "Bad Guy" to dance, because
> `electropop` contains `electro`. Both are correct under the vote.
>
> **Coverage:** 83.4% of songs land in a Billboard category, 14% are `unlabeled` and 2.5% `other`.
>
> **Known limitations, all worth stating in the blog:**
> - the labels describe the **artist**, not the song, so every song by an artist inherits the same
>   tags, and Spotify's tagging is **present-day** and retrospective — nobody called it "brill
>   building pop" in 1963
> - `unlabeled` is not evenly spread: 27.7% of 1950s songs and 39% of the partial 2020–21 window,
>   against 3.6% in the 2010s, so genre mix by decade is least reliable at both ends
> - disco tends to land in r&b rather than dance, because disco artists usually carry more soul
>   and funk labels than electronic ones
> - reggae has no Billboard genre chart in the list used here, so reggae songs fall into `other`
> - the keyword lists and the priority order are ours; a different reasonable order would move
>   songs between categories

---
## Decision 5 — the streaming-era cutoff

**Not a data question, and not a single date.** Billboard changed the Hot 100 formula repeatedly:

| Date | Change |
|---|---|
| 1991 | Nielsen SoundScan point-of-sale data replaces reported sales |
| 12 Feb 2005 | paid digital downloads counted |
| **11 Aug 2007** | streaming media and on-demand services first incorporated |
| 21 Feb 2013 | YouTube video streams added |
| 2018-2019 | paid and free streams reweighted |

Three consequences for the analysis:

1. Any "before and after streaming" gap mixes *music changing* with *the chart's measurement
   changing*. The 1991 SoundScan switch altered what charted at least as much as streaming did.
2. Billboard's **recurrent rule** removes a song from the Hot 100 once it has spent 20 weeks on
   the chart and fallen below number 50, and that rule has been revised several times. Chart
   longevity is therefore partly an administrative artefact, which matters directly for any
   claim about how long hits last.
3. A boolean forces a gradual change into a step. The trend is better shown as a continuous
   series with the dates annotated, letting a reader judge the alignment.

Dates below come from Wikipedia's Hot 100 article, which cites Billboard and a New York Times
report on the 2013 change. **Verify the 2007 and 2013 dates against Billboard's own reporting
before either appears in the blog.**

In [ ]:
# Billboard Hot 100 formula changes. Used for `streaming_era` and for annotating time-series
# figures, so that a reader sees the change was gradual rather than a single switch.
FORMULA_CHANGES = {
    "1991-01-01": "Nielsen SoundScan point-of-sale data",
    "2005-02-12": "paid digital downloads counted",
    "2007-08-11": "streaming and on-demand services first incorporated",
    "2013-02-21": "YouTube video streams added",
}

# The cutoff: Billboard's own first inclusion of streaming, not the later YouTube change.
STREAMING_ERA_START = pd.Timestamp("2007-08-11")

first_charted = billboard.groupby("song_id").chart_date.min()
streaming_era = first_charted >= STREAMING_ERA_START

print(f"cutoff: {STREAMING_ERA_START.date()} (first incorporation of streaming)\n")
print(f"songs first charting before : {(~streaming_era).sum():,}")
print(f"songs first charting after  : {streaming_era.sum():,}")

# How much data sits either side of each candidate date? A cutoff with few songs after it
# cannot support a comparison.
for date, description in FORMULA_CHANGES.items():
    after = (first_charted >= pd.Timestamp(date)).sum()
    print(f"  {date}  {after:>6,} songs after  -- {description}")

# Songs whose chart run straddles the cutoff: assigned by first chart date, as everywhere else.
straddling = billboard.groupby("song_id").chart_date.agg(["min", "max"])
crosses = ((straddling["min"] < STREAMING_ERA_START) & (straddling["max"] >= STREAMING_ERA_START))
print(f"\nsongs whose chart run crosses the cutoff: {crosses.sum()} "
      f"(assigned to the pre-streaming era by first chart date)")

> ### ✍️ Decision 5
>
> **Cutoff:** 11 August 2007, the chart date on which Billboard first incorporated streaming and
> on-demand services into the Hot 100. `streaming_era` is true for songs whose *first* chart date
> falls on or after it, consistent with how every other song-level figure is assigned.
>
> **Not used:** 21 February 2013, when YouTube was added. That date is often described as "when
> streaming came to the Hot 100", but it is five and a half years later than the actual first
> inclusion.
>
> **Source:** Wikipedia's Billboard Hot 100 article, citing Billboard and a February 2013 New York
> Times report. _(Replace with Billboard's own pages once checked.)_
>
> **How it is used:** as an annotation on continuous time series, not as the basis for a
> before-and-after claim. The boolean exists because the project brief asks for it.
>
> **What it cannot support:** any causal claim. Billboard's formula changed in 1991, 2005, 2007,
> 2013 and again in 2018-19, so a difference across the boundary mixes changes in music with
> changes in measurement. The recurrent rule, which drops songs after 20 weeks below number 50,
> makes chart longevity partly an artefact of chart administration — relevant to any finding about
> how long hits stay on the chart.